In [34]:
import pandas as pd
from sqlalchemy import create_engine
import urllib
import ast
from datetime import timedelta

In [35]:
params = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=181.57.189.150,1433;"
    "DATABASE=master;"
    "UID=sa;"
    "PWD=P@ssw0rd*;"
)

# Crear el motor de conexión
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

In [36]:
query_com = """
SELECT DISTINCT c.Cliente,
    c.Nombres + ' ' + c.Apellidos AS Nombre,
    c.Email AS Email,
    c.Celular AS Celular,
    v.Fecha AS Fecha_Venta,
    v.Canal,
    SUM(v.Venta_Neta) AS Venta
FROM Ventas_Comerssia.dbo.View_Contacto_Clientes c
INNER JOIN Ventas_Comerssia.dbo.Ventas_Unificadas v ON c.Cliente = v.Cliente
WHERE v.fecha >= '2025-07-31'
    AND v.Fecha <= '2025-07-31'
    AND Canal IN ('Shopify')
    AND v.Venta_Neta > 0
GROUP BY  c.Cliente,
         c.Documento,
       c.Nombres + ' ' + c.Apellidos,
       c.Email,
       c.Celular,
       v.Fecha,
       v.Canal
"""

# Ejecutar y cargar en DataFrame
df_ventas = pd.read_sql(query_com, engine)

df_ventas.head()

,Cliente,Nombre,Email,Celular,Fecha_Venta,Canal,Venta
0,C,JULIE ORELLANOS,NAYI39@GMAIL.COM,3044041231,2025-07-31,Shopify,5075790.17
1,C1000438338,TATIANA GOMEZ,tatianahiguita28@gmail.com,3234433835,2025-07-31,Shopify,377252.69
2,C1000506221,ANTONIO CLEVES A,acleves123@hotmail.com,3175202991,2025-07-31,Shopify,216797.40
3,C1001764478,EMMANUEL ESCOBAR MORENO,EMMA.ESMO0809@GMAIL.COM,3007586342,2025-07-31,Shopify,299987.10
4,C1001779335,WENDY MENDOZA,dywen012@gmail.com,3022069488,2025-07-31,Shopify,99575.55


In [37]:
# #consulta segmentada 
# query_data = """
# SELECT DISTINCT c.Cliente,
#     c.Documento,
#     c.Nombres + ' ' + c.Apellidos AS Nombre,
#     c.Email AS Email,
#     c.Celular AS Celular,
#     v.Fecha AS Fecha_Venta,
#     v.Canal,
#     SUM(v.Venta_Neta) AS Venta
# FROM Ventas_Comerssia.dbo.View_Contacto_Clientes c
# INNER JOIN Ventas_Comerssia.dbo.Ventas_Unificadas v ON c.Cliente = v.Cliente
# WHERE v.fecha >= '2025-07-19'
#     AND v.Fecha <= '2025-07-19'
#     AND Canal IN ('Shopify')
#     AND v.Venta_Neta > 0
#     AND v.Linea = 'ALMENDRA'
# GROUP BY  c.Cliente,
#          c.Documento,
#        c.Nombres + ' ' + c.Apellidos,
#        c.Email,
#        c.Celular,
#        v.Fecha,
#        v.Canal
# """

# # Ejecutar y cargar en DataFrame
# df_ventas = pd.read_sql(query_data, engine)

# df_ventas.head()

In [38]:
# # CONSULTAS CUMPLEAÑOS

# query_data = """
# SELECT DISTINCT v.Cliente, 
# 	cc.Nombres + ' ' +  cc.Apellidos AS Nombre, 
# 	cc.Email, cc.Celular, 
# 	v.Fecha AS Fecha_Venta 
# 	,SUM(Venta_Neta) AS Venta, 
# 	Canal
# FROM Ventas_Comerssia.dbo.Ventas_Unificadas v
# INNER JOIN Ventas_Comerssia.dbo.View_Contacto_Clientes cc ON cc.Cliente = v.Cliente
# WHERE Codigo_Descuento LIKE '%Cump%'
# 	AND Fecha BETWEEN '2025-07-04' AND '2025-07-31'
# 	AND NOT EXISTS (SELECT 1 FROM COMERSSIA.PROVENZAL.dbo.EmpleadosProvenzal e WHERE e.Codigo = v.cliente ) 
# GROUP BY  v.Cliente,
#        cc.Nombres + ' ' +  cc.Apellidos,
#        cc.Email,
#        cc.Celular,
#        v.Fecha,
#        v.Canal
# """

# # Ejecutar y cargar en DataFrame
# df_ventas = pd.read_sql(query_data, engine)
# df_ventas.head()

In [39]:
# Leer archivo
df_revie = pd.read_excel("Atribucion.xlsx")

# Seleccionar columnas relevantes
df_campania = df_revie[['Email', 'Nombre', 'Apellido', 'Phone', 'Company']].copy()

# 4. Normalizar correos
df_campania['Email'] = df_campania['Email'].str.lower().str.strip()


In [40]:
# Normalizar df_ventas
df_ventas['Email'] = df_ventas['Email'].str.lower().str.strip()
df_ventas['Fecha_Venta'] = pd.to_datetime(df_ventas['Fecha_Venta']).dt.date

# Merge por email
df_atribucion = pd.merge(df_ventas, df_campania, on='Email', how='inner')

# Sumar la venta total atribuida
total_venta = df_atribucion['Venta'].sum()

filas = len(df_atribucion) # Ver cuántas filas hay realmente

print(f"Total de Ventas: {filas}")
print(f"Total de venta atribuida: {total_venta:,.0f}"  )
# Resultado final
df_atribucion.head()

Total de Ventas: 13
Total de venta atribuida: 2,909,581


,Cliente,Nombre_x,Email,Celular,Fecha_Venta,Canal,Venta,Nombre_y,Apellido,Phone,Company
0,C1020725566,SEBASTIAN ZAMBRANO,zamsebastian@hotmail.com,3176364021,2025-07-31,Shopify,182765.25,SEBASTIAN,ZAMBRANO,3176364021,1020725566
1,C1020764976,MARÍA CAMILA GÓMEZ TRIANA,macamilagomezt@gmail.com,3017949851,2025-07-31,Shopify,141170.40,MARÍA CAMILA,GÓMEZ TRIANA,3017949851,1020764976
2,C1020824344,LAURA GUTIERRES YEPES,lauragutisports@gmail.com,3014880697,2025-07-31,Shopify,260745.09,LAURA,GUTIERRES YEPES,3014880697,1020824344
3,C1022326580,JHON JAIVER RODRIGUEZ HERRERA,jhonjv@gmail.com,3004654384,2025-07-31,Shopify,211755.60,JHON JAIVER,RODRIGUEZ HERRERA,3004654384,1022326580
4,C1032448185,VIVIANA GOMEZ ESPARZA,vivianag.esparza@gmail.com,3154997978,2025-07-31,Shopify,103987.13,VIVIANA,GOMEZ ESPARZA,3154997978,1032448185


In [41]:
df_atribucion.to_excel('Ventas.xlsx', index=False)